# Is the loss of S3 identity after IRI the collapse of the androgen program? (notebook 111)

**The question.** After IRI our mice lose the late-PT gradient (S2→S3 survival 0.31 against 0.66; notebooks 91, 105),
and the male-biased PT program falls (notebooks 91, 94). The AR-dependent male program is S2/S3-weighted (Xiong 2023).
Is the S3 identity loss partly the collapse of that program, and is it separable from a sex-independent loss of late-PT
identity (which female mice show too; Dixon 2022)?

**The frozen protocol** is `docs/results/pt-androgen-s3.md` (commit f7ce719).
- P1: AR-repressed genes rise while AR-maintained genes fall (feminisation), against gradient- and abundance-matched
  nulls and both relabelings.
- P2: does the AR-knockout log2FC predict the IR effect beyond the control gradient, abundance and transport?
- P3: Kirita timing within PTS3 nuclei (AR gap against matched S3 identity genes).
- P4: male against female in GSE144622 (24 h). P5: a glucocorticoid-response set (descriptive).

**Sections.** 1 setup; 2 sets and gradients; 3 P1; 4 P2; 5 P3; 6 P4; 7 P5 and verdicts; 8 figures; 9 run record.

## 1 · Setup and inputs

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
EXPORT = results_root / 'package_pseudospace' / 'aki' / 'pt_units.h5ad'
T91 = results_root / 'pt_positional_decomposition' / 'tables'
T76 = results_root / 'pt_package_aki' / 'tables'
T94 = results_root / 'pt_external_iri' / 'tables'
ARKO = data_root / 'external' / 'xiong2023_ar_ko'
SEXB = data_root / 'external' / 'xiong2023_sex_bias'
GSE = data_root / 'external' / 'vinas2020_gse144622'
output = results_root / 'pt_androgen_s3'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '111.androgen_s3.1'
SEED = 111
N_NULL, LFC, Q, MIN_NUCLEI, P3_STEP, P2_PERM = 2000, 1.0, .05, 30, .5, 2000
CONTROLS = ['Ctrl1A2', 'Ctrl1A4']
SEGMENTS = ['S1', 'S2', 'S3']
TIMES = ['4hours', '12hours', '2days', '14days', '6weeks']
GR_SET = ['Fkbp5', 'Tsc22d3', 'Per1', 'Sgk1', 'Zbtb16', 'Dusp1', 'Klf9', 'Errfi1']
for path in [EXPORT, T91 / 'i_genes.csv.gz', T76 / 'programs_transporters.csv', T94 / 'pseudobulk_counts.csv.gz', T94 / 'nuclei.csv',
             *[ARKO / f'ArchRpeaks_WNN_PT-{s}_mKOvsmWT_DEGs_SSeq.csv' for s in SEGMENTS],
             *[SEXB / f'PT-{s}_fWTvsmWT.csv' for s in SEGMENTS],
             *[GSE / f'GSE144622_tub.{c}.txt.gz' for c in ('IR_vs_SHAM.M', 'IR_vs_SHAM.F', 'SHAM.M_vs_F')]]:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from IPython.display import display

from pseudospace import contamination as ct
from pseudospace.external_replication import log_cpm, matched_means
from pseudospace.package_inputs import read_pt_units, support_positions
from pseudospace.regulator_activity import stratified_permutations

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
ARM_COLOR = {'AR-maintained': '#0072B2', 'AR-repressed': '#D55E00', 'identity': MUTED}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []

## 2 · AR sets, control gradient and the gene table

In [ ]:
KO = {s: pd.read_csv(ARKO / f'ArchRpeaks_WNN_PT-{s}_mKOvsmWT_DEGs_SSeq.csv').drop_duplicates('symbol').set_index('symbol') for s in SEGMENTS}
SB = {s: pd.read_csv(SEXB / f'PT-{s}_fWTvsmWT.csv').drop_duplicates('symbol').set_index('symbol') for s in SEGMENTS}
assert KO['S3'].index[0] == 'Ar' and KO['S3'].loc['Ar', 'log2FC'] < -8, 'G1: knockout table direction'
for s in SEGMENTS:
    KO[s]['tested'] = KO[s]['tested'].astype(str).str.upper().eq('TRUE')
    KO[s]['maintained'] = KO[s].tested & (KO[s].log2FC <= -LFC) & (KO[s].p_adj < Q)
    KO[s]['repressed'] = KO[s].tested & (KO[s].log2FC >= LFC) & (KO[s].p_adj < Q)
guard_log.append('G1: knockout tables read; Ar is the top S3 gene at log2FC ' + f"{KO['S3'].loc['Ar', 'log2FC']:.2f}; AR-maintained / repressed: "
                 + ', '.join(f"{s} {int(KO[s].maintained.sum())}/{int(KO[s].repressed.sum())}" for s in SEGMENTS))

G91 = pd.read_csv(T91 / 'i_genes.csv.gz').set_index('gene')
G91.index = G91.index.astype(str)

# control gradient from the AKI export controls (notebook 104's pooled estimator)
units = read_pt_units(EXPORT, 'aki')
obs = units.obs
mask, position = support_positions(obs.pseudospace.to_numpy(float), obs['sample'].astype(str).to_numpy(), obs.segment_class)
GENES = pd.Index(units.var_names.astype(str))
counts = sparse.csr_matrix(units.X[mask])
spec = obs['sample'].astype(str).to_numpy()[mask]
del units
E_ctrl = pd.Series(np.mean([ct.end_contrast(counts[np.flatnonzero(spec == s)], position[spec == s]) for s in CONTROLS], axis=0), GENES)

tr = pd.read_csv(T76 / 'programs_transporters.csv')
TPT = set(tr.mouse[tr.in_T_PT.astype(bool)].astype(str))
D = pd.DataFrame(index=G91.index.intersection(KO['S3'].index[KO['S3'].tested]))
D['L'] = G91.loc[D.index, 'L']
D['L R1'] = G91.loc[D.index, 'L R1']
D['L R2'] = G91.loc[D.index, 'L R2']
D['Δ late'] = G91.loc[D.index, 'Δ late']
D['level'] = G91.loc[D.index, 'log2 CP10k (controls)']
D['E ctrl'] = E_ctrl.reindex(D.index)
D['TPT'] = D.index.isin(TPT)
D['ko S3'] = KO['S3'].loc[D.index, 'log2FC']
for s in SEGMENTS:
    D[f'maintained {s}'] = KO[s].maintained.reindex(D.index).fillna(False).astype(bool)
    D[f'repressed {s}'] = KO[s].repressed.reindex(D.index).fillna(False).astype(bool)
D = D.dropna(subset=['L', 'E ctrl', 'level'])
D['stratum'] = (pd.qcut(D['E ctrl'].rank(method='first'), 10, labels=False) * 10
                + pd.qcut(D['level'].rank(method='first'), 10, labels=False)).astype(int)
D.to_csv(tables / 'sets.csv')
display(pd.DataFrame({k: [int(D[k].sum())] for k in [c for c in D.columns if c.startswith(('maintained', 'repressed'))]}, index=['in universe']))
guard_log.append(f'G2: {len(D):,} genes in notebook 91 ∩ S3 knockout tested; S3 AR-maintained {int(D["maintained S3"].sum())}, '
                 f'AR-repressed {int(D["repressed S3"].sum())}')

## 3 · P1 · feminisation

In [ ]:
def set_test(members, cols, seed):
    pool = D[cols].to_numpy()
    obs_ = D.loc[members, cols].mean().to_numpy()
    null = matched_means(pool, D.stratum.to_numpy(), D.loc[members, 'stratum'].to_numpy(), n=N_NULL, seed=seed)
    return obs_, null


rows = []
for seg in ('S3', 'S2'):
    for arm, col in (('AR-maintained', f'maintained {seg}'), ('AR-repressed', f'repressed {seg}')):
        members = D.index[D[col]]
        obs_, null = set_test(members, ['L', 'L R1', 'L R2', 'Δ late'], SEED + len(rows))
        rows.append({'segment': seg, 'arm': arm, 'genes': len(members), 'mean L': obs_[0], 'mean L R1': obs_[1], 'mean L R2': obs_[2],
                     'mean Δ late': obs_[3], 'null L 5%': float(np.quantile(null[:, 0], .05)), 'null L mean': float(null[:, 0].mean()),
                     'null L 95%': float(np.quantile(null[:, 0], .95)), 'null Δ late mean': float(null[:, 3].mean())})
P1 = pd.DataFrame(rows)
r3 = P1[P1.segment.eq('S3')].set_index('arm')
DEREPRESS = bool(r3.loc['AR-repressed', 'mean L'] > r3.loc['AR-repressed', 'null L 95%']
                 and r3.loc['AR-repressed', 'mean L'] > max(r3.loc['AR-repressed', 'mean L R1'], r3.loc['AR-repressed', 'mean L R2']))
AR_LOSS = bool(r3.loc['AR-maintained', 'mean L'] < r3.loc['AR-maintained', 'null L 5%']
               and r3.loc['AR-maintained', 'mean L'] < min(r3.loc['AR-maintained', 'mean L R1'], r3.loc['AR-maintained', 'mean L R2']))
FEMINISE = DEREPRESS and AR_LOSS
P1.to_csv(tables / 'p1.csv', index=False)
display(P1.round(3))
print(f'P1: AR loss {AR_LOSS}; de-repression {DEREPRESS}; feminisation {FEMINISE}')
for arm, col in (('AR-maintained', 'maintained S3'), ('AR-repressed', 'repressed S3')):
    top = D[D[col]].sort_values('L')
    print(arm, 'lowest L:', ', '.join(f'{g} {v:.2f}' for g, v in top.L.head(8).items()), '| highest L:',
          ', '.join(f'{g} {v:.2f}' for g, v in top.L.tail(8).items()))

## 4 · P2 · AR dependence beyond the gradient, abundance and transport

In [ ]:
def ko_coef(y):
    Z = np.column_stack([np.ones(len(D)), D['E ctrl'], D['level'], D['TPT'].astype(float)])
    x = D['ko S3'].to_numpy(float)
    Q_, _ = np.linalg.qr(Z)
    ry = y - Q_ @ (Q_.T @ y)
    rx = x - Q_ @ (Q_.T @ x)
    coef = float(rx @ ry / (rx @ rx))
    perms = stratified_permutations(D.stratum.to_numpy(), P2_PERM, SEED + 7)
    Xp = x[perms].T
    Rx = Xp - Q_ @ (Q_.T @ Xp)
    null = (Rx * ry[:, None]).sum(axis=0) / (Rx ** 2).sum(axis=0)
    return coef, float((1 + (null >= coef).sum()) / (P2_PERM + 1)), null


P2_ROWS = []
for outcome in ('L', 'L R1', 'L R2'):
    coef, p, null = ko_coef(D[outcome].to_numpy(float))
    P2_ROWS.append({'outcome': outcome, 'knockout coefficient': coef, 'p (> 0)': p, 'null 95%': float(np.quantile(null, .95))})
P2 = pd.DataFrame(P2_ROWS)
P2.to_csv(tables / 'p2.csv', index=False)
P2_HOLDS = bool(P2.iloc[0]['knockout coefficient'] > 0 and P2.iloc[0]['p (> 0)'] <= .05
                and P2.iloc[0]['knockout coefficient'] > P2.iloc[1:]['knockout coefficient'].abs().max())
display(P2.round(4))
print('P2 holds:', P2_HOLDS)

## 5 · P3 · Kirita timing within PTS3 nuclei

In [ ]:
PB = pd.read_csv(T94 / 'pseudobulk_counts.csv.gz', index_col=0)
NUC = pd.read_csv(T94 / 'nuclei.csv').set_index('mouse')
pts3 = PB[[c for c in PB.columns if c.endswith('|PTS3')]]
pts3.columns = pts3.columns.str.rsplit('|', n=1).str[0]
pts1 = PB[[c for c in PB.columns if c.endswith('|PTS1')]]
pts1.columns = pts1.columns.str.rsplit('|', n=1).str[0]
keep = [m for m in pts3.columns if NUC.loc[m, 'PTS3'] >= MIN_NUCLEI]
LC3 = pd.DataFrame(log_cpm(pts3[keep].to_numpy()), index=pts3.index, columns=keep)
ctrl_mice = [m for m in keep if m.startswith('Control|')]
LC1c = pd.DataFrame(log_cpm(pts1[[m for m in pts1.columns if m.startswith('Control|')]].to_numpy()), index=pts1.index).mean(axis=1)
ctrl3 = LC3[ctrl_mice].mean(axis=1)
dep_any = pd.concat([(KO[s].log2FC.abs() >= .5) | ~KO[s].tested for s in SEGMENTS], axis=1).any(axis=1)
sex_any = pd.concat([SB[s].p_adj < Q for s in SEGMENTS], axis=1).any(axis=1)
genes_k = pts3.index.astype(str)
ar_set = pd.Index([g for g in KO['S3'].index[KO['S3'].maintained] if g in genes_k])
identity = pd.Index([g for g in genes_k if (ctrl3.get(g, -np.inf) - LC1c.get(g, np.inf) >= 1)
                     and g in KO['S3'].index and not dep_any.get(g, True) and not sex_any.get(g, False)])
dec = pd.qcut(ctrl3.rank(method='first'), 10, labels=False)
w = dec.reindex(ar_set).value_counts(normalize=True)
p3_rows = []
for t in TIMES:
    mice = [m for m in keep if m.startswith(t + '|')]
    if len(mice) == 0:
        p3_rows.append({'time': t, 'mice': 0})
        continue
    change = LC3[mice].mean(axis=1) - ctrl3
    ar = float(change.reindex(ar_set).mean())
    id_by_dec = change.reindex(identity).groupby(dec.reindex(identity)).mean()
    idc = float(sum(w.get(d, 0) * id_by_dec.get(d, np.nan) for d in w.index if d in id_by_dec.index) / sum(w.get(d, 0) for d in w.index if d in id_by_dec.index))
    p3_rows.append({'time': t, 'mice': len(mice), 'AR-maintained change': ar, 'identity change': idc, 'AR gap': ar - idc,
                    'PTS3 nuclei': ', '.join(str(int(NUC.loc[m, 'PTS3'])) for m in mice)})
P3 = pd.DataFrame(p3_rows)
P3.to_csv(tables / 'p3_kirita.csv', index=False)
display(P3.round(3))
gap = P3.set_index('time')['AR gap']
P3_HOLDS = bool(min(gap.get('12hours', np.nan), gap.get('2days', np.nan)) <= gap.get('4hours', np.nan) - P3_STEP
                and gap.get('14days', np.nan) <= -P3_STEP)
print(f'P3: AR set {len(ar_set)} genes, identity set {len(identity)} genes; mice kept {len(keep)}/{pts3.shape[1]}; holds {P3_HOLDS}')
guard_log.append(f'G3: Kirita PTS3 pseudobulk, {len(keep)} of {pts3.shape[1]} mice with ≥ {MIN_NUCLEI} PTS3 nuclei; '
                 f'AR-maintained S3 {len(ar_set)} genes, identity {len(identity)}')

## 6 · P4 · male against female (GSE144622, 24 h)

In [ ]:
def read_gse(c):
    t = pd.read_csv(GSE / f'GSE144622_tub.{c}.txt.gz', sep='\t')
    t = t.dropna(subset=['mgi_symbol']).drop_duplicates('mgi_symbol').set_index('mgi_symbol')
    return t


GM, GF, GS = read_gse('IR_vs_SHAM.M'), read_gse('IR_vs_SHAM.F'), read_gse('SHAM.M_vs_F')
assert GS.loc['Ddx3y', 'log2FoldChange'] > 3, 'G4: sign of the sex contrast'
common = GM.index.intersection(GF.index)
common = common[GM.loc[common, 'log2FoldChange'].notna() & GF.loc[common, 'log2FoldChange'].notna()]
gdec = pd.qcut(GM.loc[common, 'baseMean'].rank(method='first'), 10, labels=False)
p4_rows = []
for seg in ('S3', 'S2'):
    for arm, flag in (('AR-maintained', 'maintained'), ('AR-repressed', 'repressed')):
        members = [g for g in KO[seg].index[KO[seg][flag]] if g in common]
        vals = np.column_stack([GM.loc[common, 'log2FoldChange'], GF.loc[common, 'log2FoldChange']])
        null = matched_means(vals, gdec.to_numpy(), gdec.reindex(members).to_numpy(), n=N_NULL, seed=SEED + 40)
        obs_m, obs_f = GM.loc[members, 'log2FoldChange'].mean(), GF.loc[members, 'log2FoldChange'].mean()
        p4_rows.append({'segment': seg, 'arm': arm, 'genes': len(members), 'male lfc': obs_m, 'male null 5%': float(np.quantile(null[:, 0], .05)),
                        'male null mean': float(null[:, 0].mean()), 'female lfc': obs_f, 'female null 5%': float(np.quantile(null[:, 1], .05)),
                        'female null mean': float(null[:, 1].mean()),
                        'DiD': float((obs_m - null[:, 0].mean()) - (obs_f - null[:, 1].mean()))})
P4 = pd.DataFrame(p4_rows)
P4.to_csv(tables / 'p4_gse144622.csv', index=False)
r = P4[(P4.segment == 'S3') & (P4.arm == 'AR-maintained')].iloc[0]
P4_HOLDS = bool(r['male lfc'] < r['male null 5%'] and not r['female lfc'] < r['female null 5%'])
display(P4.round(3))
print('P4 male-specific collapse:', P4_HOLDS)
guard_log.append(f'G4: GSE144622 sign checked (Ddx3y +{GS.loc["Ddx3y", "log2FoldChange"]:.2f} male-high); {len(common):,} genes in both IR contrasts')

## 7 · P5 and verdicts

In [ ]:
gr = [g for g in GR_SET if g in G91.index]
P5 = G91.loc[gr, ['L', 'L R1', 'L R2', 'Δ late']]
P5.to_csv(tables / 'p5_gr.csv')
display(P5.round(3))
VERDICTS = pd.DataFrame([
    ('P1 AR-maintained S3 genes fall (below matched null and relabelings)', f"{AR_LOSS} · mean L {r3.loc['AR-maintained', 'mean L']:.3f} (null {r3.loc['AR-maintained', 'null L mean']:.3f}; R1 {r3.loc['AR-maintained', 'mean L R1']:.3f}, R2 {r3.loc['AR-maintained', 'mean L R2']:.3f})"),
    ('P1 AR-repressed S3 genes rise (de-repression)', f"{DEREPRESS} · mean L {r3.loc['AR-repressed', 'mean L']:.3f} (null {r3.loc['AR-repressed', 'null L mean']:.3f}; R1 {r3.loc['AR-repressed', 'mean L R1']:.3f}, R2 {r3.loc['AR-repressed', 'mean L R2']:.3f})"),
    ('P1 feminisation', str(FEMINISE)),
    ('P2 knockout log2FC predicts L beyond gradient, abundance, transport', f"{P2_HOLDS} · coef {P2.iloc[0]['knockout coefficient']:.4f} (p {P2.iloc[0]['p (> 0)']:.3g}); relabelings {P2.iloc[1]['knockout coefficient']:.4f}, {P2.iloc[2]['knockout coefficient']:.4f}"),
    ('P3 Kirita AR gap grows after 4 h and persists at 14 d', f"{P3_HOLDS} · gaps " + ', '.join(f'{t} {v:.2f}' for t, v in gap.items() if np.isfinite(v))),
    ('P4 male-specific collapse (GSE144622)', f"{P4_HOLDS} · male {r['male lfc']:.3f} (null mean {r['male null mean']:.3f}), female {r['female lfc']:.3f} (null mean {r['female null mean']:.3f}), DiD {r['DiD']:.3f}"),
    ('P5 glucocorticoid-response genes, L', ', '.join(f'{g} {v:.2f}' for g, v in P5.L.items())),
], columns=['reading', 'result'])
VERDICTS.to_csv(tables / 'verdicts.csv', index=False)
display(VERDICTS)

## 8 · Figures (plots only)

In [ ]:
fig, ax = plt.subplots(figsize=(80 * MM, 45 * MM))
for i, (_, row) in enumerate(P1.iterrows()):
    ax.hlines(i, row['null L 5%'], row['null L 95%'], color=LIGHT, lw=3)
    ax.scatter(row['mean L'], i, color=ARM_COLOR[row.arm], s=14, zorder=3)
    ax.scatter([row['mean L R1'], row['mean L R2']], [i, i], color=ARM_COLOR[row.arm], s=8, marker='x', zorder=3)
ax.axvline(0, color=MUTED, lw=.5)
ax.set_yticks(range(len(P1)), [f'{a} {s}' for a, s in zip(P1.arm, P1.segment)])
ax.set_xlabel('mean IR level change (log2)')
save(fig, '111a_ar_sets')
plt.show()

fig, ax = plt.subplots(figsize=(70 * MM, 55 * MM))
sc = ax.scatter(D['ko S3'], D['L'], c=D['E ctrl'].clip(-3, 3), cmap='RdBu_r', s=2, lw=0)
ax.axhline(0, color=LIGHT, lw=.5); ax.axvline(0, color=LIGHT, lw=.5)
ax.set_xlabel('Ar knockout log2FC, S3'); ax.set_ylabel('IR level change (log2)')
plt.colorbar(sc, ax=ax, shrink=.7)
save(fig, '111b_ko_vs_L')
plt.show()

fig, ax = plt.subplots(figsize=(70 * MM, 42 * MM))
tt = P3.dropna(subset=['AR gap'])
ax.plot(range(len(tt)), tt['AR-maintained change'], marker='o', color=ARM_COLOR['AR-maintained'])
ax.plot(range(len(tt)), tt['identity change'], marker='o', color=ARM_COLOR['identity'])
ax.plot(range(len(tt)), tt['AR gap'], marker='s', color=INK, ls=':')
ax.axhline(0, color=LIGHT, lw=.5)
ax.set_xticks(range(len(tt)), tt.time)
ax.set_ylabel('log2 change in PTS3 nuclei')
save(fig, '111c_kirita_timing')
plt.show()

fig, ax = plt.subplots(figsize=(60 * MM, 55 * MM))
both = [g for g in KO['S3'].index[KO['S3'].maintained] if g in common]
ax.scatter(GM.loc[common, 'log2FoldChange'], GF.loc[common, 'log2FoldChange'], s=1, color=LIGHT, lw=0)
ax.scatter(GM.loc[both, 'log2FoldChange'], GF.loc[both, 'log2FoldChange'], s=4, color=ARM_COLOR['AR-maintained'], lw=0)
ax.axhline(0, color=MUTED, lw=.4); ax.axvline(0, color=MUTED, lw=.4)
ax.set_xlabel('IR vs sham, male (log2)'); ax.set_ylabel('IR vs sham, female (log2)')
save(fig, '111d_male_female')
plt.show()

## 9 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'E_ctrl is the mean over the two control mice of the pooled end contrast (x ≥ 0.7 against x ≤ 0.3) on the AKI export common support.'),
    ('reading', 'The P1 and P2 strata are the joint deciles of E_ctrl and control log2 CP10k over notebook 91\'s genes that the S3 knockout table tested.'),
    ('reading', 'P3 uses notebook 94\'s per-mouse PTS3 pseudobulks with voom-style log2 CPM; the identity change is weighted to the AR set\'s decile distribution of control PTS3 expression; genes untested in any knockout segment are excluded from the identity set.'),
    ('reading', 'P4 matches on the decile of baseMean in the male IR-vs-sham table, over genes with a fold change in both sex contrasts.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '111_pt_androgen_s3', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'p1': P1.round(4).to_dict('records'), 'feminise': FEMINISE, 'ar_loss': AR_LOSS, 'derepress': DEREPRESS,
       'p2': P2.round(4).to_dict('records'), 'p2_holds': P2_HOLDS, 'p3': P3.round(4).to_dict('records'), 'p3_holds': P3_HOLDS,
       'p4': P4.round(4).to_dict('records'), 'p4_holds': P4_HOLDS, 'p5': P5.round(3).reset_index().to_dict('records'),
       'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps({k: RUN[k] for k in ('ar_loss', 'derepress', 'feminise', 'p2_holds', 'p3_holds', 'p4_holds', 'seconds')}, indent=2))